# InternVL2.5-2B -- Positional Bias Pipeline

Same pipeline structure as the PaliGemma 2 / Kosmos-2 / MolmoE notebooks
(mount Drive, load samples, run zero-shot, save resumable per-dataset
results), adapted for `OpenGVLab/InternVL2_5-2B` (bbox-tier: free-text
chat model with a documented visual-grounding prompt format, ~2B params).

InternVL2.5-2B weights are open (not gated) on Hugging Face, so no HF
token/login is required to download it. It does require `trust_remote_code=True`
(the model ships custom modeling code, not a native `transformers` class).

In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Define the root directory for sample datasets on Google Drive
SAMPLES_ROOT = Path("/content/drive/MyDrive/samples")

# Define the root directory where results will be saved
RESULTS_ROOT = Path("/content/drive/MyDrive/results/internvl2_5-2b")

# Create the results directory if it doesn't exist
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Samples root:", SAMPLES_ROOT)
print("Samples root exists:", SAMPLES_ROOT.exists())

print("\nAvailable datasets:")
for folder in SAMPLES_ROOT.iterdir():
    if folder.is_dir():
        print("-", folder.name)

In [ ]:
!pip install -U transformers accelerate sentencepiece einops timm

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected!")

In [ ]:
import csv
import os
import re
from pathlib import Path

from PIL import Image

import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

# Define the model ID for InternVL2.5-2B
MODEL_ID = "OpenGVLab/InternVL2_5-2B"
# Define the datasets to be used
DATASETS = ("lgg", "kvasir", "isic")

In [ ]:
# --- InternVL's documented image preprocessing (dynamic tiling) ---
# Copied from the model card's standard usage snippet: InternVL2.5 splits a
# high-resolution image into up to `max_num` 448x448 tiles (plus a resized
# thumbnail of the whole image) so the vision encoder sees fine detail
# without just downsampling everything to 448x448. This does NOT change how
# bbox coordinates are interpreted later: InternVL2.5 always reports box
# coordinates normalized to a 0-1000 scale of the ORIGINAL image, regardless
# of how many tiles were used internally.

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1)
        for j in range(1, n + 1) if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images


def load_pixel_values(image: Image.Image, input_size=448, max_num=12, dtype=torch.bfloat16):
    """Turns a PIL image into the (n_tiles, 3, 448, 448) pixel_values tensor
    InternVL2.5's model.chat() expects, via the dynamic-tiling preprocessing
    documented on the model card."""
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values.to(dtype)

In [ ]:
def to_internvl_prompt(prompt_text: str) -> str:
    """
    Rewrites the manifest's "Please locate ..." instruction into InternVL2.5's
    documented visual-grounding chat format -- InternVL2.5 was trained to
    respond to grounding requests phrased as:
        'Please provide the bounding box coordinate of the region this
        sentence describes: <ref>{phrase}</ref>'
    with a reply containing '<box>[[x1, y1, x2, y2]]</box>' (coordinates
    normalized to a 0-1000 scale of the ORIGINAL image), same judgment call
    already made for Kosmos-2 and PaliGemma: verify it actually respects the
    positional wording (left/right/upper/lower) placed inside <ref>, rather
    than ignoring everything after the object name.

    'Please locate the tumor on the left side of this image.'
    -> 'Please provide the bounding box coordinate of the region this
        sentence describes: <ref>the tumor on the left side of this
        image</ref>'
    """
    s = prompt_text.strip()
    if s.lower().startswith("please locate "):
        s = s[len("please locate "):]
    phrase = s.rstrip(". ").strip()
    return (
        "Please provide the bounding box coordinate of the region this "
        f"sentence describes: <ref>{phrase}</ref>"
    )


_BOX_RE = re.compile(r"<box>\s*\[\[\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\]\]\s*</box>")


def parse_bbox_from_output(generated_text: str, img_w: int, img_h: int):
    """
    InternVL2.5's grounding output looks like:
        '<ref>the tumor...</ref><box>[[123, 456, 234, 567]]</box>'
    with all four numbers on a 0-1000 normalized scale of the ORIGINAL image
    (not the resized/tiled model input), regardless of how many tiles
    dynamic_preprocess() used. Returns a pixel-space (xmin, ymin, xmax, ymax)
    tuple, or None if no box was found in the reply.
    """
    match = _BOX_RE.search(generated_text)
    if match is None:
        return None
    x0, y0, x1, y1 = (int(v) for v in match.groups())
    return (
        x0 / 1000 * img_w,
        y0 / 1000 * img_h,
        x1 / 1000 * img_w,
        y1 / 1000 * img_h,
    )


def bbox_center_and_side(bbox, img_w, img_h):
    """
    Calculate the center coordinates and determine the side (left/right, upper/lower)
    of a bounding box within an image. Same function used by the other model pipelines.
    """
    xmin, ymin, xmax, ymax = bbox
    cx = (xmin + xmax) / 2
    cy = (ymin + ymax) / 2
    side_lr = "left" if cx < img_w / 2 else "right"
    side_ud = "upper" if cy < img_h / 2 else "lower"
    return cx, cy, side_lr, side_ud

In [ ]:
def load_model():
    """
    Loads InternVL2.5-2B and its tokenizer, optimizing for GPU if available.
    """
    if torch.cuda.is_available():
        gpu_capability = torch.cuda.get_device_capability()[0]
        if gpu_capability >= 8:
            dtype = torch.bfloat16
            print("Using bfloat16")
        else:
            dtype = torch.float16
            print("Using float16")
        device = "cuda:0"
    else:
        dtype = torch.float32
        device = "cpu"
        torch.set_num_threads(os.cpu_count() or 1)
        print("WARNING: No GPU detected. Running on CPU.")

    model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(device)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_fast=False,
    )

    model.eval()
    return tokenizer, model, device, dtype

In [ ]:
# Load the model and tokenizer using the defined function
tokenizer, model, device, dtype = load_model()

print("Model loaded successfully!")
print("Device:", device, "| dtype:", dtype)

In [ ]:
def run_one(tokenizer, model, device, dtype, image: Image.Image, question: str) -> str:
    """
    Runs InternVL2.5-2B on a single image, asking it the grounding question
    built by to_internvl_prompt(), via the model's documented .chat() API.

    Returns the raw generated text (still containing the <ref>/<box> tags,
    if any -- parsed separately by parse_bbox_from_output()).
    """
    pixel_values = load_pixel_values(image, dtype=dtype).to(device)
    generation_config = dict(max_new_tokens=256, do_sample=False)

    with torch.no_grad():
        response = model.chat(tokenizer, pixel_values, question, generation_config)

    return response

In [ ]:
def run_dataset(dataset_name, tokenizer, model, device, dtype, limit=None):
    """
    Runs InternVL2.5-2B over a dataset, resuming from any existing results
    file and flushing after every row so a Colab disconnect/interrupt loses
    at most the row in progress, not everything since the last full save.
    """
    sample_dir = SAMPLES_ROOT / dataset_name
    manifest_path = sample_dir / f"manifest_{dataset_name}.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found:\n{manifest_path}")

    with manifest_path.open() as f:
        rows = list(csv.DictReader(f))
    if limit is not None:
        rows = rows[:limit]

    out_path = RESULTS_ROOT / f"results_internvl2_5-2b_{dataset_name}.csv"

    fieldnames = [
        "model", "dataset", "sample_id", "prompt_id", "prompt_text", "model_prompt",
        "raw_output", "pred_bbox_xmin", "pred_bbox_ymin", "pred_bbox_xmax", "pred_bbox_ymax",
        "pred_centroid_x", "pred_centroid_y", "pred_side_lr", "pred_side_ud",
        "gt_centroid_x", "gt_centroid_y", "gt_side_lr", "gt_side_ud",
    ]

    # Resume support: skip (sample_id, prompt_id) pairs already saved from a
    # previous run of this exact dataset, instead of overwriting from scratch.
    done = set()
    file_exists = out_path.exists() and out_path.stat().st_size > 0
    if file_exists:
        with out_path.open() as f:
            for r in csv.DictReader(f):
                done.add((r["sample_id"], r["prompt_id"]))
        print(f"[resume] found {len(done)} already-completed rows in {out_path.name}")

    rows_to_run = [r for r in rows if (r["sample_id"], r["prompt_id"]) not in done]

    print(f"Dataset: {dataset_name}")
    print(f"Total rows: {len(rows)} | already done: {len(done)} | remaining: {len(rows_to_run)}")

    mode = "a" if file_exists else "w"
    with out_path.open(mode, newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
            f.flush()

        for i, row in enumerate(rows_to_run, 1):
            img_path = sample_dir / row["image_path"]
            image = Image.open(img_path).convert("RGB")
            img_w, img_h = image.size

            model_prompt = to_internvl_prompt(row["prompt_text"])
            raw_output = run_one(tokenizer, model, device, dtype, image, model_prompt)
            bbox_px = parse_bbox_from_output(raw_output, img_w, img_h)

            if bbox_px is None:
                pred_cx = pred_cy = pred_side_lr = pred_side_ud = ""
            else:
                pred_cx, pred_cy, pred_side_lr, pred_side_ud = bbox_center_and_side(bbox_px, img_w, img_h)

            writer.writerow({
                "model": "internvl2_5-2b",
                "dataset": dataset_name,
                "sample_id": row["sample_id"],
                "prompt_id": row["prompt_id"],
                "prompt_text": row["prompt_text"],
                "model_prompt": model_prompt,
                "raw_output": raw_output,
                "pred_bbox_xmin": bbox_px[0] if bbox_px else "",
                "pred_bbox_ymin": bbox_px[1] if bbox_px else "",
                "pred_bbox_xmax": bbox_px[2] if bbox_px else "",
                "pred_bbox_ymax": bbox_px[3] if bbox_px else "",
                "pred_centroid_x": pred_cx,
                "pred_centroid_y": pred_cy,
                "pred_side_lr": pred_side_lr,
                "pred_side_ud": pred_side_ud,
                "gt_centroid_x": row["gt_centroid_x"],
                "gt_centroid_y": row["gt_centroid_y"],
                "gt_side_lr": row["gt_side_lr"],
                "gt_side_ud": row["gt_side_ud"],
            })
            f.flush()  # ensures this row survives a disconnect
            status = "no-detection" if bbox_px is None else f"{pred_side_lr}/{pred_side_ud}"
            print(f"[{dataset_name}] {i}/{len(rows_to_run)} | {row['sample_id']} | {row['prompt_id']} -> {status}")

    print("\nCompleted!")
    print(f"Results saved to:\n{out_path}")
    return out_path

## Full runs (resumable -- safe to re-run if interrupted)

In [ ]:
result_path = run_dataset(
    dataset_name="lgg",
    tokenizer=tokenizer,
    model=model,
    device=device,
    dtype=dtype
)

In [ ]:
result_path = run_dataset(
    dataset_name="kvasir",
    tokenizer=tokenizer,
    model=model,
    device=device,
    dtype=dtype
)

In [ ]:
result_path = run_dataset(
    dataset_name="isic",
    tokenizer=tokenizer,
    model=model,
    device=device,
    dtype=dtype
)

## Visualize predictions

Saves one combined PNG per sample -- original image | ground truth mask (red) | visual check (gt bbox/centroid + midlines, red) | one prediction panel per prompt P0-P4 (blue) -- all side by side, directly to `RESULTS_ROOT/<dataset>/predicted_visual_check/<sample_id>.png`. Nothing is displayed inline; browse the folder afterward to compare every prompt's prediction for a given image at a glance.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # no inline display -- this cell only writes PNG files to disk
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def visualize_sample_all_prompts(dataset_name, results_csv_path, sample_id):
    """
    Saves (does NOT display) ONE combined PNG per sample_id, with every
    prompt's prediction side by side so they're easy to compare at a glance:
      1. Original image
      2. Ground truth -- the REAL segmentation mask overlaid on the image
         (translucent red) plus its centroid, straight from the dataset's
         mask file (never from the model).
      3. Visual check -- same style as scripts/visualize_ground_truth.py:
         the ground-truth bbox/centroid (red) drawn on the plain image, with
         yellow midline/midheight reference lines and a side_lr/side_ud
         label, so the left/right/upper/lower call can be sanity-checked
         directly against the drawn box instead of the mask overlay.
      4..N. Prediction -- this model's predicted bbox (blue) and centroid
         for each prompt_id found in results_csv_path for this sample,
         in prompt_id order (P0-P4), one panel per prompt.

    Written to RESULTS_ROOT/<dataset_name>/predicted_visual_check/
    <sample_id>.png -- so you can browse every (input image, ground truth,
    visual check, all-prompts prediction) row as a single file later,
    without needing to re-run any cells or scroll back through notebook
    output.
    """
    sample_dir = SAMPLES_ROOT / dataset_name
    manifest_path = sample_dir / f"manifest_{dataset_name}.csv"

    with manifest_path.open() as f:
        manifest_row = next(r for r in csv.DictReader(f) if r["sample_id"] == sample_id)

    image = Image.open(sample_dir / manifest_row["image_path"]).convert("RGB")
    mask = Image.open(sample_dir / manifest_row["mask_path"]).convert("L")
    img_w, img_h = image.size

    with Path(results_csv_path).open() as f:
        sample_rows = sorted(
            (r for r in csv.DictReader(f) if r["sample_id"] == sample_id),
            key=lambda r: r["prompt_id"],
        )

    n_panels = 3 + len(sample_rows)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

    # Panel 1: original image, untouched
    axes[0].imshow(image)
    axes[0].set_title(f"{sample_id}\nOriginal image")
    axes[0].axis("off")

    # Panel 2: ground truth -- real mask overlaid + centroid
    axes[1].imshow(image)
    mask_arr = np.array(mask)
    overlay = np.zeros((*mask_arr.shape, 4))
    overlay[mask_arr > 127] = [1, 0, 0, 0.4]  # translucent red where mask is foreground
    axes[1].imshow(overlay)
    gt_cx = float(sample_rows[0]["gt_centroid_x"])
    gt_cy = float(sample_rows[0]["gt_centroid_y"])
    axes[1].plot(gt_cx, gt_cy, "ro", markersize=8)
    axes[1].set_title(f"Ground truth\n{sample_rows[0]['gt_side_lr']}/{sample_rows[0]['gt_side_ud']}")
    axes[1].axis("off")

    # Panel 3: visual check -- gt bbox + centroid + midlines on the plain
    # image, matching scripts/visualize_ground_truth.py's annotation style,
    # so left/right/upper/lower can be sanity-checked against the drawn box.
    axes[2].imshow(image)
    axes[2].axvline(img_w / 2, color="yellow", linewidth=1)
    axes[2].axhline(img_h / 2, color="yellow", linewidth=1)
    gx0 = float(manifest_row["gt_bbox_xmin"])
    gy0 = float(manifest_row["gt_bbox_ymin"])
    gx1 = float(manifest_row["gt_bbox_xmax"])
    gy1 = float(manifest_row["gt_bbox_ymax"])
    axes[2].add_patch(patches.Rectangle((gx0, gy0), gx1 - gx0, gy1 - gy0, linewidth=2, edgecolor="red", facecolor="none"))
    axes[2].plot(gt_cx, gt_cy, "ro", markersize=8)
    axes[2].set_title(f"GT Sanity Check (bbox + midline)\n{sample_rows[0]['gt_side_lr']}/{sample_rows[0]['gt_side_ud']}")
    axes[2].axis("off")

    # Panels 4..N: one per prompt -- bbox + centroid (no mask: this model is
    # scored at the bbox/centroid level, see the run_one()/run_dataset() note)
    for ax, result_row in zip(axes[3:], sample_rows):
        ax.imshow(image)
        if result_row["pred_bbox_xmin"]:
            x0, y0 = float(result_row["pred_bbox_xmin"]), float(result_row["pred_bbox_ymin"])
            x1, y1 = float(result_row["pred_bbox_xmax"]), float(result_row["pred_bbox_ymax"])
            rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="blue", facecolor="none")
            ax.add_patch(rect)
            pred_cx, pred_cy = float(result_row["pred_centroid_x"]), float(result_row["pred_centroid_y"])
            ax.plot(pred_cx, pred_cy, "bo", markersize=8)
            pred_label = f"{result_row['pred_side_lr']}/{result_row['pred_side_ud']}"
        else:
            pred_label = "no-detection"
        ax.set_title(f"{result_row['prompt_id']}: {result_row['prompt_text']}\n{pred_label}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()

    viz_dir = RESULTS_ROOT / dataset_name / "predicted_visual_check"
    viz_dir.mkdir(parents=True, exist_ok=True)
    out_file = viz_dir / f"{sample_id}.png"
    plt.savefig(out_file, dpi=100, bbox_inches="tight")
    plt.close(fig)
    return out_file

def visualize_dataset_all_prompts(dataset_name):
    """
    Saves one combined PNG per sample (original | ground truth | visual
    check | every prompt's prediction, side by side) for the WHOLE dataset
    -- no inline display, just files on disk under
    RESULTS_ROOT/<dataset_name>/predicted_visual_check/.
    """
    result_path = RESULTS_ROOT / f"results_internvl2_5-2b_{dataset_name}.csv"

    with result_path.open() as f:
        sample_ids = sorted(set(r["sample_id"] for r in csv.DictReader(f)))

    for i, sample_id in enumerate(sample_ids, 1):
        out_file = visualize_sample_all_prompts(dataset_name, result_path, sample_id)
        if i % 10 == 0 or i == len(sample_ids):
            print(f"[{dataset_name}] {i}/{len(sample_ids)} | saved -> {out_file}")

    print(f"\nDone. All {len(sample_ids)} samples saved to "
          f"{(RESULTS_ROOT / dataset_name / 'predicted_visual_check')}")

for dataset_name in DATASETS:
    visualize_dataset_all_prompts(dataset_name)